# 技术特征、聚类与价格标签

运行顺序：01_residual_dynamics.ipynb → 本Notebook → 03_incremental_prediction.ipynb。

读取完整行情与上一步的data/ar1/signals.csv，连续计算历史特征，只交付2023–2025。按原来源阶段过滤跨界标签窗：train为2023–2024，validation为2025。这些是来源名称，分类建模阶段随后合并并重新按Y分层随机60/20/20。

本Notebook只做技术特征、资格筛选、历史聚类、同组偏离、事件标签与数据交付。不做单变量显著性、Bootstrap、策略诊断或模型训练。

## 1. 配置、行情与残差动力学输入

保留2022年预热；不使用2026年。原始价格、质量字段及AR有效性标记保持可追溯。

In [1]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import silhouette_score

DATA_DIR = Path('data')
AR_PATH = DATA_DIR / 'ar1' / 'signals.csv'
BENCHMARK_CODES = {'510300', '510500', '159915', '588000', '513100', '518880'}
cfg = {
    'analysis_start': '2023-02-02', 'train_end': '2024-12-31',
    'validation_start': '2025-01-01', 'test_end': '2025-12-31',
    'price_window': 20, 'label_horizon': 20, 'contraction_alpha': 0.5,
    'trigger_sigma': 1.25, 'cooldown_sessions': 20,
    'min_listed_days': 250, 'min_adv20': 30000000,
    'cluster_lookback': 250, 'cluster_rebalance_days': 60,
    'cluster_min_observations': 200, 'cluster_min_pair_observations': 160,
    'cluster_k_min': 2, 'cluster_k_max': 12
}
panel = pd.read_parquet(DATA_DIR / 'etf_daily_full_coverage_483.parquet')
panel['date'] = pd.to_datetime(panel['date'])
panel['code'] = panel['code'].astype('string').str.zfill(6)
panel = panel.loc[panel['date'].le(cfg['test_end'])].sort_values(['code', 'date']).reset_index(drop=True)
if panel.duplicated(['date', 'code']).any():
    raise ValueError('行情存在重复date/code')
panel['split'] = 'warmup'
panel.loc[panel['date'].between(cfg['analysis_start'], cfg['train_end']), 'split'] = 'train'
panel.loc[panel['date'].between(cfg['validation_start'], cfg['test_end']), 'split'] = 'validation'

ar = pd.read_csv(AR_PATH, dtype={'code': 'string'}, encoding='utf-8-sig')
for col in ['date', 'fit_start', 'fit_end']:
    ar[col] = pd.to_datetime(ar[col], format='mixed')
ar = ar.rename(columns={'alpha': 'ols_alpha'})
ar = ar.loc[ar['date'].le(cfg['test_end'])].copy()
AR_COLUMNS = ['ols_alpha', 'beta_hs300', 'beta_zz500', 'beta_cyb', 'beta_kc50',
              'beta_oversea', 'beta_gold', 'eps', 'X', 'kappa', 'm', 'sigma_eq', 'HL', 's',
              'ols_r2', 'fit_start', 'fit_end', 'ols_valid', 'ar1_valid', 'signal_valid',
              'signal_status', 'is_benchmark', 'price_input_policy']
if ar.duplicated(['date', 'code']).any() or ar['fit_end'].ge(ar['date']).any():
    raise ValueError('AR输入主键重复或拟合使用了当日/未来数据')
formal_keys = panel.loc[panel['split'].ne('warmup'), ['date', 'code']]
missing_ar = formal_keys.merge(ar[['date', 'code']], on=['date', 'code'], how='left', indicator=True)
if missing_ar['_merge'].ne('both').any():
    raise ValueError('日频AR输入缺少正式期间记录，请先完整运行01_residual_dynamics.ipynb')
panel[['date', 'code', 'close_hfq', 'split']].head()

/var/folders/1n/4m_zt2yn3jq9xkrxx1894sdc0000gp/T/ipykernel_3303/1232440036.py:32: DtypeWarning: Columns (0: fit_start, 1: fit_end) have mixed types. Specify dtype option on import or set low_memory=False.
  ar = pd.read_csv(AR_PATH, dtype={'code': 'string'}, encoding='utf-8-sig')


,date,code,close_hfq,split
0,2022-01-04,159001,100.001,warmup
1,2022-01-05,159001,100.000,warmup
2,2022-01-06,159001,100.002,warmup
3,2022-01-07,159001,100.000,warmup
4,2022-01-10,159001,100.000,warmup


## 2. 技术特征与资格筛选

收益采用对数收益，vol_20采用20日样本标准差；RSI为14日对数收益正负部分的简单均值比，MACD采用12/26/9日EMA。已核实停牌日成交额按0计入ADV20。

资格要求历史不少于250日、ADV20不少于3000万元、当日可交易、非填充/停牌，排除六基准。成交额保留原值并附log1p列；分类白名单仍由分类建模阶段决定。

In [2]:
def grouped_roll(series: pd.Series, groups: pd.Series, window: int, op: str, min_periods: int | None = None) -> pd.Series:
    min_periods = window if min_periods is None else min_periods
    rolled = series.groupby(groups, sort=False).rolling(window, min_periods=min_periods)
    return getattr(rolled, op)().reset_index(level=0, drop=True).reindex(series.index)

def build_technical_features(panel: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    out = panel.copy()
    code = out["code"]
    price = out["close_hfq"].where(out["close_hfq"] > 0)
    logp = np.log(price)
    by_code = logp.groupby(code, sort=False)
    out["ret_1"] = by_code.diff(1)
    out["ret_5"] = logp - by_code.shift(5)
    out["ret_20"] = logp - by_code.shift(20)
    out["vol_20"] = grouped_roll(out["ret_1"], code, 20, "std")

    liquidity_amount = out["amount"].where(~out["verified_suspension"].fillna(False), 0.0)
    out["adv20_calc"] = grouped_roll(liquidity_amount, code, 20, "mean", 20)
    # 成交额逐行固定变换，不用全样本估计统计量。
    out["log_amount"] = np.log1p(out["amount"].clip(lower=0))
    out["log_adv20_calc"] = np.log1p(out["adv20_calc"].clip(lower=0))

    gains = out["ret_1"].clip(lower=0)
    losses = (-out["ret_1"].clip(upper=0))
    avg_gain = grouped_roll(gains, code, 14, "mean", 14)
    avg_loss = grouped_roll(losses, code, 14, "mean", 14)
    rs = avg_gain / avg_loss.replace(0, np.nan)
    out["rsi_14"] = 100 - 100 / (1 + rs)
    out.loc[(avg_loss == 0) & (avg_gain > 0), "rsi_14"] = 100.0
    out.loc[(avg_loss == 0) & (avg_gain == 0), "rsi_14"] = 50.0

    def ewm(span: int, source: pd.Series) -> pd.Series:
        return source.groupby(code, sort=False).transform(
            lambda s: s.ewm(span=span, adjust=False, min_periods=span).mean()
        )

    ema12 = ewm(12, price)
    ema26 = ewm(26, price)
    out["macd"] = ema12 - ema26
    out["macd_signal"] = ewm(9, out["macd"])
    out["macd_hist"] = out["macd"] - out["macd_signal"]
    out["ma_20"] = grouped_roll(price, code, 20, "mean", 20)
    out["bias_20"] = price / out["ma_20"] - 1.0

    tradable = out["is_tradable"].fillna(False)
    imputed = out["is_imputed"].fillna(False)
    suspended = out["verified_suspension"].fillna(False) | out["is_suspended"].fillna(False)
    out["roleb_pass"] = (
        price.notna()
        & out["listed_days"].ge(cfg["min_listed_days"])
        & out["adv20_calc"].ge(cfg["min_adv20"])
        & tradable & ~imputed & ~suspended
        & ~out["code"].isin(BENCHMARK_CODES)
    )
    return out

features = build_technical_features(panel, cfg)
features[['date', 'code', 'ret_20', 'vol_20', 'rsi_14', 'adv20_calc', 'roleb_pass']].tail()

,date,code,ret_20,vol_20,rsi_14,adv20_calc,roleb_pass
468022,2025-12-25,588400,0.072639,0.017200,60.285714,43578480.45,True
468023,2025-12-26,588400,0.067021,0.017230,52.617845,43645535.05,True
468024,2025-12-29,588400,0.049217,0.017136,48.187477,42887838.55,True
468025,2025-12-30,588400,0.066437,0.017200,51.971697,43394947.00,True
468026,2025-12-31,588400,0.051917,0.017940,51.130567,43913735.05,True


## 3. 过去250日收益聚类

在快照日按当日资格选候选，仅用快照日前250日收益计算相关性；距离1−rho，average linkage层次聚类，按silhouette最高选2–12组，并列选较小k。每60个交易日更新，不使用未来标签。

In [3]:
def build_clusters(features: pd.DataFrame, cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    returns = features.pivot(index="date", columns="code", values="ret_1").sort_index()
    eligible = features.set_index(["date", "code"])["roleb_pass"]
    dates = returns.index
    analysis_position = int(dates.searchsorted(pd.Timestamp(cfg["analysis_start"])))
    start = max(cfg["cluster_lookback"], analysis_position)
    test_end = pd.Timestamp(cfg["test_end"])
    snapshots = [
        pd.Timestamp(dates[i])
        for i in range(start, len(dates), cfg["cluster_rebalance_days"])
        if pd.Timestamp(dates[i]) <= test_end
    ]
    maps, diag = [], []
    for snap in snapshots:
        pos = int(dates.searchsorted(snap))
        history = returns.iloc[pos - cfg["cluster_lookback"]:pos]
        try:
            valid_now = eligible.xs(snap, level="date")
        except KeyError:
            continue
        candidates = valid_now[valid_now].index.intersection(history.columns)
        counts = history[candidates].notna().sum()
        candidates = counts[counts >= cfg["cluster_min_observations"]].index
        if len(candidates) < 6:
            diag.append({"snapshot_date": snap, "status": "insufficient_etfs", "n_etfs": len(candidates)})
            continue
        corr = history[candidates].corr(min_periods=cfg["cluster_min_pair_observations"])
        corr = corr.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-1.0, 1.0)
        values = corr.to_numpy(copy=True)
        np.fill_diagonal(values, 1.0)
        distance = np.clip(1.0 - values, 0.0, 2.0)
        tree = linkage(squareform(distance, checks=False), method="average")
        best = None
        max_k = min(cfg["cluster_k_max"], len(candidates) - 1)
        for k in range(cfg["cluster_k_min"], max_k + 1):
            labels = fcluster(tree, k, criterion="maxclust")
            if len(np.unique(labels)) < 2:
                continue
            score = silhouette_score(distance, labels, metric="precomputed")
            if best is None or (score, -k) > (best[0], -best[1]):
                best = (score, k, labels)
        if best is None:
            diag.append({"snapshot_date": snap, "status": "no_valid_k", "n_etfs": len(candidates)})
            continue
        score, k, labels = best
        local = pd.DataFrame({"snapshot_date": snap, "code": candidates.astype("string"), "cluster_local": labels.astype(int)})
        local["group_size"] = local.groupby("cluster_local")["code"].transform("size").astype(int)
        local["cluster_id"] = snap.strftime("%Y%m%d") + "_C" + local["cluster_local"].astype(str).str.zfill(2)
        maps.append(local)
        diag.append({
            "snapshot_date": snap, "status": "ok", "n_etfs": len(candidates),
            "n_clusters": k, "silhouette": score,
            "min_group_size": int(local["group_size"].min()),
            "median_group_size": float(local["group_size"].median()),
            "max_group_size": int(local["group_size"].max()),
        })
    return (pd.concat(maps, ignore_index=True) if maps else pd.DataFrame()), pd.DataFrame(diag)

cluster_map, cluster_summary = build_clusters(features, cfg)
if cluster_map.empty:
    raise ValueError('正式期间未形成有效聚类快照')
cluster_summary

,snapshot_date,status,n_etfs,n_clusters,silhouette,min_group_size,median_group_size,max_group_size
0,2023-02-02,ok,193,2,0.513086,13,180.0,180
1,2023-04-28,ok,221,2,0.475032,18,203.0,203
2,2023-07-28,ok,199,2,0.431898,16,183.0,183
3,2023-10-30,ok,188,2,0.450414,20,168.0,168
4,2024-01-23,ok,183,2,0.451549,11,172.0,172
5,2024-04-26,ok,185,2,0.472092,11,174.0,174
6,2024-07-25,ok,176,2,0.485457,12,164.0,164
7,2024-10-28,ok,256,2,0.598355,13,243.0,243
8,2025-01-21,ok,200,2,0.566162,12,188.0,188
9,2025-04-24,ok,205,2,0.550488,13,192.0,192


## 4. 同组偏离

沿用最近的历史快照成员，不每日重新按资格筛组员。group_dev为20日对数收益减同组其他成员均值，group_size为快照组规模；无同组其他有效成员时保持缺失，不用未来快照回填。ret_1/ret_5偏离仅为中间输出，不进入当前C白名单。

In [4]:
def attach_group_features(features: pd.DataFrame, cluster_map: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    out = features.copy()
    out["cluster_id"] = pd.Series(pd.NA, index=out.index, dtype="string")
    out["cluster_snapshot_date"] = pd.NaT
    out["group_size"] = np.nan
    snaps = sorted(pd.to_datetime(cluster_map["snapshot_date"].unique())) if not cluster_map.empty else []
    terminal = pd.Timestamp(cfg["test_end"]) + pd.Timedelta(days=1)
    for i, snap in enumerate(snaps):
        next_snap = snaps[i + 1] if i + 1 < len(snaps) else terminal
        mapping = cluster_map.loc[cluster_map["snapshot_date"].eq(snap)].set_index("code")
        mask = out["date"].ge(snap) & out["date"].lt(next_snap)
        codes = out.loc[mask, "code"]
        out.loc[mask, "cluster_id"] = codes.map(mapping["cluster_id"]).astype("string").to_numpy()
        out.loc[mask, "cluster_snapshot_date"] = snap
        out.loc[mask, "group_size"] = codes.map(mapping["group_size"]).to_numpy()
    keys = [out["date"], out["cluster_id"]]
    for source, target in [("ret_1", "group_dev_1"), ("ret_5", "group_dev_5"), ("ret_20", "group_dev")]:
        valid = out[source].notna() & out["cluster_id"].notna()
        sums = out[source].where(valid).groupby(keys, dropna=False).transform("sum")
        counts = out[source].where(valid).groupby(keys, dropna=False).transform("count")
        peers = counts - 1
        out[target] = out[source] - (sums - out[source]) / peers.replace(0, np.nan)
        out.loc[~valid | peers.le(0), target] = np.nan
        if target == "group_dev":
            out["group_obs_n"] = peers.where(valid & peers.gt(0))
    return out

features = attach_group_features(features, cluster_map, cfg)
features[['date', 'code', 'cluster_snapshot_date', 'group_dev', 'group_size']].tail()

,date,code,cluster_snapshot_date,group_dev,group_size
468022,2025-12-25,588400,2025-10-23,0.046623,248.0
468023,2025-12-26,588400,2025-10-23,0.041812,248.0
468024,2025-12-29,588400,2025-10-23,0.037327,248.0
468025,2025-12-30,588400,2025-10-23,0.044189,248.0
468026,2025-12-31,588400,2025-10-23,0.026475,248.0


## 5. 价格偏离事件与标签

事件日偏离为ln(P/MA20)，触发阈值为其20日滚动样本标准差的1.25倍。逐ETF连续冷却20交易日，不在年份边界重置；先进行冷却抽样，再排除跨来源阶段或观察不足的窗口。

Y=1：未来20个交易日内，至少一个可交易且非人工填充的观测，其相对事件日冻结MA的绝对对数偏离≤初始绝对偏离的50%。Y_abs是另一固定0.5sigma带标签，不供分类建模阶段使用。未来字段只用于标签，不入模。

In [5]:
def build_events(features: pd.DataFrame, cfg: dict) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rows, paths = [], []
    w, n = cfg["price_window"], cfg["label_horizon"]
    alpha, trigger = cfg["contraction_alpha"], cfg["trigger_sigma"]
    cooldown = cfg["cooldown_sessions"]
    for code, g in features.groupby("code", sort=False, observed=True):
        g = g.sort_values("date").reset_index(drop=True)
        p = g["close_hfq"].to_numpy(float)
        date = g["date"].to_numpy(dtype="datetime64[ns]")
        split = g["split"].astype(str).to_numpy()
        eligible = g["roleb_pass"].fillna(False).to_numpy(bool)
        evidence_ok = (g["is_tradable"].fillna(False) & ~g["is_imputed"].fillna(False)).to_numpy(bool)
        ma = pd.Series(p).rolling(w, min_periods=w).mean().to_numpy()
        d = np.log(p / ma)
        sigma = pd.Series(d).rolling(w, min_periods=w).std(ddof=1).to_numpy()
        trigger_positions = np.flatnonzero(
            eligible & np.isfinite(d) & np.isfinite(sigma) & (sigma > 0)
            & (np.abs(d) >= trigger * sigma) & np.isin(split, ["train", "validation"])
        )
        last_accepted = -100000
        for pos in trigger_positions:
            accepted = pos > last_accepted + cooldown
            if accepted:
                last_accepted = pos
            end = pos + n
            complete = end < len(g)
            same_stage = complete and split[end] == split[pos]
            future_prices = p[pos + 1:end + 1] if complete else np.array([], dtype=float)
            future_evidence = evidence_ok[pos + 1:end + 1] if complete else np.array([], dtype=bool)
            price_complete = complete and np.isfinite(future_prices).all() and (future_prices > 0).all()
            label_ok = bool(accepted and same_stage and price_complete)
            row = {
                "date": pd.Timestamp(date[pos]), "code": str(code), "split": split[pos],
                "MA_T": ma[pos], "d_T": d[pos], "sigma_d_T": sigma[pos],
                "theta_trig": trigger * sigma[pos], "trigger_direction": int(np.sign(d[pos])),
                "price_window": w, "label_window": n, "contraction_alpha": alpha,
                "trigger_sigma": trigger, "cooldown_sessions": cooldown,
                "is_nonoverlap_event": bool(accepted), "label_window_complete": bool(complete),
                "label_same_stage": bool(same_stage), "label_window_ok_20": label_ok,
                "label_end_date_20": pd.Timestamp(date[end]) if complete else pd.NaT,
            }
            if label_ok:
                future_abs = np.abs(np.log(future_prices / ma[pos]))
                hit = (future_abs <= alpha * abs(d[pos])) & future_evidence
                abs_hit = (future_abs <= 0.5 * sigma[pos]) & future_evidence
                row["Y"] = int(hit.any())
                row["Y_abs"] = int(abs_hit.any())
                row["days_to_revert"] = int(np.argmax(hit) + 1) if hit.any() else np.nan
                row["ret_fwd_N"] = math.log(p[end] / p[pos])
                for k, ratio in enumerate(future_abs / abs(d[pos]), start=1):
                    paths.append({"date": pd.Timestamp(date[pos]), "code": str(code), "split": split[pos], "Y": row["Y"], "k": k, "abs_dev_ratio": ratio})
            else:
                row.update({"Y": np.nan, "Y_abs": np.nan, "days_to_revert": np.nan, "ret_fwd_N": np.nan})
            rows.append(row)
    all_events = pd.DataFrame(rows)
    if not all_events.empty:
        all_events["event_id"] = all_events["date"].dt.strftime("%Y%m%d") + "_" + all_events["code"]
    events = all_events.loc[all_events["label_window_ok_20"]].copy().reset_index(drop=True)
    events["Y"] = events["Y"].astype("int8")
    events["Y_abs"] = events["Y_abs"].astype("int8")
    return events, all_events, pd.DataFrame(paths)

events, event_candidates, event_paths = build_events(features, cfg)
events.groupby('split')['Y'].agg(['size', 'mean'])

,size,mean
split,,
train,3640,0.639560
validation,1988,0.549296


## 6. 合并A/B/C并检查交付

完整日频表按date/code合并AR输出，事件表再按键取特征。B无效或C缺失的行保留，让分类建模阶段按统一A/B白名单筛共同样本；不使用额外log波动、log半衰期或s有效标记扩大删行范围。

日频表与事件表保存至原有路径，覆盖前应确认正在使用同一版行情和参数。重新生成后源CSV指纹可能因精度/列格式改变，旧模型仍对应旧输入；需要重跑分类建模阶段才能取得本次数据的实验结果。

In [6]:
daily = features.merge(ar[['date', 'code', *AR_COLUMNS]], on=['date', 'code'], how='left', validate='one_to_one')
feature_columns = [col for col in daily.columns if col not in events.columns or col in ['date', 'code']]
event_features = events.merge(daily[feature_columns], on=['date', 'code'], how='left', validate='one_to_one')
if event_features.duplicated(['date', 'code']).any() or event_features['Y'].isna().any():
    raise ValueError('事件主键重复或标签缺失')
if not event_features['label_same_stage'].all() or not event_features['label_window_complete'].all():
    raise ValueError('事件含跨来源阶段或不完整标签窗')
if (daily['cluster_snapshot_date'] > daily['date']).any():
    raise ValueError('组特征使用了未来快照')
delivery_summary = event_features.groupby('split').agg(
    events=('Y', 'size'), etfs=('code', 'nunique'), success_rate=('Y', 'mean'),
    C_available=('group_dev', 'count'))
delivery_summary

,events,etfs,success_rate,C_available
split,,,,
train,3640,287,0.639560,3343
validation,1988,296,0.549296,1753


## 7. 保存真实数据

导出train（2023–2024）与validation（2025）两个来源CSV。它们是上游交付阶段，不限制分类建模阶段后续重新随机划分。不会生成策略、Bootstrap或额外报告文件。

In [7]:
for folder in ['daily', 'events']:
    (DATA_DIR / folder).mkdir(parents=True, exist_ok=True)
for split in ['train', 'validation']:
    daily.loc[daily['split'].eq(split)].to_csv(
        DATA_DIR / 'daily' / f'{split}.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    event_features.loc[event_features['split'].eq(split)].to_csv(
        DATA_DIR / 'events' / f'{split}.csv', index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
delivery_summary

,events,etfs,success_rate,C_available
split,,,,
train,3640,287,0.639560,3343
validation,1988,296,0.549296,1753


## 最终数据表与 Summary

`final_daily_data` 是正式交付的完整日频表，`final_event_data` 是完成20日标签计算后的事件表，均与上面的导出内容一致。分别展示前10行的主要字段，完整列保留在变量中。这里的 train / validation 是上游来源阶段，分类建模阶段 会重新随机划分。

先查看日频A/B/C特征，再查看事件触发信息、标签和标签窗结束日期。Summary 同时报告样本覆盖、标签比例及关键特征缺失；本阶段保留的缺失值不代表已经填补。

In [ ]:
# 去除仅用于历史计算的预热行，保留正式导出阶段。
final_daily_data = daily.loc[daily['split'].isin(['train', 'validation'])].reset_index(drop=True)
final_event_data = event_features.reset_index(drop=True)
pd.set_option('display.max_columns', 30)
final_daily_data[['date', 'code', 'split', 'close_hfq', 'ret_1', 'ret_5', 'ret_20',
                  'vol_20', 'log_amount', 'log_adv20_calc', 'bias_20',
                  'm', 'sigma_eq', 'HL', 'cluster_id', 'group_dev', 'group_size',
                  'roleb_pass']].head(10)

,date,code,split,close_hfq,ret_1,ret_5,ret_20,vol_20,log_amount,log_adv20_calc,bias_20,m,sigma_eq,HL,cluster_id,group_dev,group_size,roleb_pass
0,2023-02-02,159001,train,100.000,0.00000,0.00000,-0.00001,0.000010,20.538590,20.294879,-1.499998e-06,5.088592e-06,0.000010,1.009538,20230202_C02,-0.081376,180.0,True
1,2023-02-03,159001,train,100.000,0.00000,0.00000,-0.00002,0.000010,20.396221,20.316657,-4.999998e-07,-2.017195e-07,0.000009,0.729993,20230202_C02,-0.076628,180.0,True
2,2023-02-06,159001,train,100.000,0.00000,-0.00001,0.00000,0.000009,19.888193,20.229997,-4.999998e-07,5.509300e-07,0.000009,0.725556,20230202_C02,-0.059800,180.0,True
3,2023-02-07,159001,train,100.000,0.00000,0.00001,-0.00001,0.000008,20.395074,20.259235,0.000000e+00,4.339781e-06,0.000010,0.979003,20230202_C02,-0.053238,180.0,True
4,2023-02-08,159001,train,100.001,0.00001,0.00001,0.00000,0.000009,20.104471,20.259202,1.000000e-05,-6.792197e-07,0.000009,0.714124,20230202_C02,-0.042811,180.0,True
5,2023-02-09,159001,train,99.997,-0.00004,-0.00003,-0.00003,0.000012,20.089930,20.236564,-2.850004e-05,-5.304477e-06,0.000010,1.018794,20230202_C02,-0.043354,180.0,True
6,2023-02-10,159001,train,100.000,0.00003,0.00000,0.00000,0.000014,20.851016,20.287452,1.500002e-06,1.394457e-05,0.000011,1.307761,20230202_C02,-0.033570,180.0,True
7,2023-02-13,159001,train,100.000,0.00000,0.00000,0.00000,0.000014,20.053057,20.282152,1.500002e-06,-7.312603e-07,0.000009,0.774522,20230202_C02,-0.032497,180.0,True
8,2023-02-14,159001,train,99.999,-0.00001,-0.00001,-0.00001,0.000014,19.869958,20.284408,-8.000016e-06,-6.523450e-06,0.000009,0.662929,20230202_C02,-0.030216,180.0,True
9,2023-02-15,159001,train,100.001,0.00002,0.00000,0.00001,0.000015,19.895737,20.271200,1.150002e-05,-2.274488e-07,0.000009,0.711250,20230202_C02,-0.029446,180.0,True


In [ ]:
# Y 为未来20日内是否达到偏离收缩目标，标签窗只用于构造标签。
final_event_data[['date', 'code', 'split', 'close_hfq', 'MA_T', 'd_T',
                  'theta_trig', 'trigger_direction', 'm', 'sigma_eq', 'HL',
                  'cluster_id', 'group_dev', 'group_size', 'Y', 'Y_abs',
                  'days_to_revert', 'label_end_date_20']].head(10)

,date,code,split,close_hfq,MA_T,d_T,theta_trig,trigger_direction,m,sigma_eq,HL,cluster_id,group_dev,group_size,Y,Y_abs,days_to_revert,label_end_date_20
0,2023-02-08,159001,train,100.001,100.00000,0.000010,0.000008,1,-6.792197e-07,0.000009,0.714124,20230202_C02,-0.042811,180.0,1,1,2.0,2023-03-08
1,2023-03-15,159001,train,99.996,99.99965,-0.000037,0.000029,-1,3.824825e-06,0.000011,0.536633,20230202_C02,0.044854,180.0,1,1,1.0,2023-04-13
2,2023-05-10,159001,train,99.999,100.00045,-0.000015,0.000011,-1,-8.934065e-06,0.000015,0.785060,20230428_C02,0.052478,203.0,1,0,1.0,2023-06-07
3,2023-06-26,159001,train,100.002,100.00060,0.000014,0.000008,1,-7.062502e-06,0.000014,0.552498,20230428_C02,0.000795,203.0,1,0,2.0,2023-07-24
4,2023-08-15,159001,train,99.999,100.00040,-0.000014,0.000007,-1,-8.606524e-06,0.000014,0.664391,20230728_C02,0.013045,183.0,1,0,1.0,2023-09-12
5,2023-09-13,159001,train,99.999,100.00020,-0.000012,0.000008,-1,1.276003e-05,0.000013,0.420372,20230728_C02,0.030657,183.0,1,1,2.0,2023-10-19
6,2023-11-16,159001,train,100.002,100.00065,0.000013,0.000007,1,5.484248e-06,0.000017,1.150868,20231030_C02,-0.004764,20.0,1,0,1.0,2023-12-14
7,2023-12-28,159001,train,99.997,100.00015,-0.000032,0.000011,-1,1.240998e-05,0.000014,0.610548,20231030_C02,-0.003643,20.0,1,1,1.0,2024-01-26
8,2024-02-06,159001,train,99.999,100.00050,-0.000015,0.000012,-1,1.051180e-05,0.000020,1.719662,20240123_C01,-0.003906,11.0,1,0,1.0,2024-03-13
9,2024-03-21,159001,train,100.002,100.00045,0.000015,0.000009,1,-9.134231e-06,0.000019,2.115713,20240123_C01,-0.002902,11.0,1,0,1.0,2024-04-22


In [ ]:
# Summary：日频表和事件表分别统计，避免混淆两种样本单位。
final_roleb_summary = pd.concat([
    frame.groupby('split', sort=False).agg(
        rows=('code', 'size'), etfs=('code', 'nunique'), dates=('date', 'nunique'),
        date_start=('date', 'min'), date_end=('date', 'max')
    ).assign(table=name, columns=frame.shape[1]).reset_index()
    for name, frame in [('daily', final_daily_data), ('events', final_event_data)]
], ignore_index=True)
final_roleb_summary[['table', 'split', 'rows', 'columns', 'etfs', 'dates', 'date_start', 'date_end']]

,table,split,rows,columns,etfs,dates,date_start,date_end
0,daily,train,225561,109,483,467,2023-02-02,2024-12-31
1,daily,validation,117369,109,483,243,2025-01-02,2025-12-31
2,events,train,3640,129,287,424,2023-02-02,2024-12-03
3,events,validation,1988,129,296,207,2025-01-02,2025-12-03


In [ ]:
# 标签与特征覆盖：分组特征允许暂时缺失，交给 分类建模阶段 的训练集填补器处理。
final_label_summary = final_event_data.groupby('split', sort=False).agg(
    events=('Y', 'size'), positive_events=('Y', 'sum'), success_rate=('Y', 'mean'),
    label_missing=('Y', lambda s: s.isna().sum()),
    group_dev_missing=('group_dev', lambda s: s.isna().sum()),
    group_size_missing=('group_size', lambda s: s.isna().sum()))
final_label_summary['B_incomplete_events'] = final_event_data[['m', 'sigma_eq', 'HL']].isna().any(axis=1).groupby(final_event_data['split'], sort=False).sum()
final_label_summary

,events,positive_events,success_rate,label_missing,group_dev_missing,group_size_missing,B_incomplete_events
split,,,,,,,
train,3640,2328,0.639560,0,297,297,1
validation,1988,1092,0.549296,0,235,235,0


In [ ]:
# 逐项检查交付事件表中实际参与建模的特征及标签。
final_feature_columns = ['ret_1', 'ret_5', 'ret_20', 'vol_20', 'rsi_14', 'macd',
                         'macd_signal', 'macd_hist', 'log_amount', 'log_adv20_calc',
                         'bias_20', 'd_T', 'm', 'sigma_eq', 'HL', 'group_dev', 'group_size', 'Y']
final_roleb_feature_summary = final_event_data[final_feature_columns].describe().T
final_roleb_feature_summary['missing_count'] = final_event_data[final_feature_columns].isna().sum()
final_roleb_feature_summary['missing_rate'] = final_event_data[final_feature_columns].isna().mean()
final_roleb_feature_summary

,count,mean,std,min,25%,50%,75%,max,missing_count,missing_rate
ret_1,5628.0,0.002008,0.029133,-0.202657,-0.012516,0.000027,0.013542,1.167292,0,0.000000
ret_5,5628.0,0.005615,0.064885,-0.691964,-0.026956,0.000390,0.028584,2.304422,0,0.000000
ret_20,5628.0,0.012948,0.097735,-2.316146,-0.036997,0.001749,0.051066,2.306329,0,0.000000
vol_20,5628.0,0.014077,0.013703,0.000005,0.008930,0.012520,0.017302,0.516277,0,0.000000
rsi_14,5628.0,52.134990,20.213094,0.470237,34.997137,54.399831,68.518355,100.000000,0,0.000000
macd,5628.0,0.008064,0.437409,-24.213393,-0.006616,0.001220,0.014163,21.035180,0,0.000000
macd_signal,5628.0,0.006299,0.306742,-18.857668,-0.005726,0.001377,0.012980,9.528278,0,0.000000
macd_hist,5628.0,0.001765,0.172925,-5.355725,-0.002796,-0.000033,0.003379,11.506902,0,0.000000
log_amount,5628.0,18.849450,1.342742,13.893295,17.839838,18.580176,19.651041,24.594816,0,0.000000
log_adv20_calc,5628.0,18.751141,1.309004,17.216931,17.706520,18.433656,19.539962,24.299001,0,0.000000
